In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np
import cv2

In [2]:
dataset, info = tfds.load('oxford_iiit_pet', with_info=True)
print(info)

tfds.core.DatasetInfo(
    name='oxford_iiit_pet',
    full_name='oxford_iiit_pet/4.0.0',
    description="""
    The Oxford-IIIT pet dataset is a 37 category pet image dataset with roughly 200
    images for each class. The images have large variations in scale, pose and
    lighting. All images have an associated ground truth annotation of breed and
    species. Additionally, head bounding boxes are provided for the training split,
    allowing using this dataset for simple object detection tasks. In the test
    split, the bounding boxes are empty.
    """,
    homepage='http://www.robots.ox.ac.uk/~vgg/data/pets/',
    data_dir='/root/tensorflow_datasets/oxford_iiit_pet/4.0.0',
    file_format=tfrecord,
    download_size=773.52 MiB,
    dataset_size=773.68 MiB,
    features=FeaturesDict({
        'file_name': Text(shape=(), dtype=string),
        'head_bbox': BBoxFeature(shape=(4,), dtype=float32),
        'image': Image(shape=(None, None, 3), dtype=uint8),
        'label': ClassLab

In [3]:
train_data = dataset['train']
test_data = dataset['test']

IMG_SIZE = 128
BATCH_SIZE = 16

In [ ]:
def preprocess(data):
    image = tf.image.resize(data['image'], (IMG_SIZE, IMG_SIZE))
    mask = tf.image.resize(data['segmentation_mask'], (IMG_SIZE, IMG_SIZE), method='nearest')
    image = tf.cast(image, tf.float32) / 255
    mask = tf.cast(mask, tf.int32) - 1
    return image, mask

In [11]:
train_ds=(train_data.map(preprocess).cache().shuffle(1000).batch(BATCH_SIZE))

Tensor("resize_1/Squeeze:0", shape=(128, 128, 1), dtype=uint8)


In [13]:
def conv_block(x, filters):
    x = keras.layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = keras.layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    return x

In [14]:
def encoder_block(x, filters):
    f = conv_block(x, filters)
    p = keras.layers.MaxPool2D((2, 2))(f)
    return f, p

In [15]:
def decoder_block(x, skip, filters):
    x = keras.layers.Conv2DTranspose(filters, (2, 2), strides=2, padding='same')(x)
    x = keras.layers.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x

In [17]:
def build_unet(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=3):
    inputs = keras.layers.Input(input_shape)

    f1, p1 = encoder_block(inputs, 64)
    f2, p2 = encoder_block(p1, 128)
    f3, p3 = encoder_block(p2, 256)
    f4, p4 = encoder_block(p3, 512)

    bottleneck = conv_block(p4, 1024)

    d1 = decoder_block(bottleneck, f4, 512)
    d2 = decoder_block(d1, f3, 256)
    d3 = decoder_block(d2, f2, 128)
    d4 = decoder_block(d3, f1, 64)

    outputs = keras.layers.Conv2D(3, (1, 1), activation='softmax')(d4)

    model = keras.Model(inputs, outputs, name='U-Net')
    return model

In [19]:
model = build_unet()
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "U-Net"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 128, 128,  │      1,792 │ input_layer_1[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 128, 128,  │     36,928 │ conv2d_19[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 64, 64,    │          0 │ conv2d_20[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 64, 64,    │     73,856 │ max_pooling2d_4[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_22 (Conv2D)  │ (None, 64, 64,    │    147,584 │ conv2d_21[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 32, 32,    │          0 │ conv2d_22[0][0]   │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_23 (Conv2D)  │ (None, 32, 32,    │    295,168 │ max_pooling2d_5[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_24 (Conv2D)  │ (None, 32, 32,    │    590,080 │ conv2d_23[0][0]   │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_6     │ (None, 16, 16,    │          0 │ conv2d_24[0][0]   │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_25 (Conv2D)  │ (None, 16, 16,    │  1,180,160 │ max_pooling2d_6[… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_26 (Conv2D)  │ (None, 16, 16,    │  2,359,808 │ conv2d_25[0][0]   │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_7     │ (None, 8, 8, 512) │          0 │ conv2d_26[0][0]   │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_27 (Conv2D)  │ (None, 8, 8,      │  4,719,616 │ max_pooling2d_7[… │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_28 (Conv2D)  │ (None, 8, 8,      │  9,438,208 │ conv2d_27[0][0]   │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_4  │ (None, 16, 16,    │  2,097,664 │ conv2d_28[0][0]   │
│ (Conv2DTranspose)   │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 16, 16,    │          0 │ conv2d_transpose

 Total params: 31,031,875 (118.38 MB)

 Trainable params: 31,031,875 (118.38 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
history = model.fit(train_ds, epochs=10)

Epoch 1/10
230/230 ━━━━━━━━━━━━━━━━━━━━ 110s 246ms/step - accuracy: 0.5847 - loss: 0.9132
Epoch 2/10
230/230 ━━━━━━━━━━━━━━━━━━━━ 56s 245ms/step - accuracy: 0.7218 - loss: 0.6759
Epoch 3/10
230/230 ━━━━━━━━━━━━━━━━━━━━ 57s 246ms/step - accuracy: 0.7679 - loss: 0.5837
Epoch 4/10
230/230 ━━━━━━━━━━━━━━━━━━━━ 56s 245ms/step - accuracy: 0.8048 - loss: 0.5019
Epoch 5/10
230/230 ━━━━━━━━━━━━━━━━━━━━ 56s 244ms/step - accuracy: 0.8260 - loss: 0.4482
Epoch 6/10
230/230 ━━━━━━━━━━━━━━━━━━━━ 56s 244ms/step - accuracy: 0.8332 - loss: 0.4312
Epoch 7/10
230/230 ━━━━━━━━━━━━━━━━━━━━ 56s 244ms/step - accuracy: 0.8483 - loss: 0.3944
Epoch 8/10
230/230 ━━━━━━━━━━━━━━━━━━━━ 56s 246ms/step - accuracy: 0.8579 - loss: 0.3739
Epoch 9/10
230/230 ━━━━━━━━━━━━━━━━━━━━ 57s 246ms/step - accuracy: 0.8695 - loss: 0.3427
Epoch 10/10
230/230 ━━━━━━━━━━━━━━━━━━━━ 56s 244ms/step - accuracy: 0.8766 - loss: 0.3238


In [21]:
def create_mask(pred_mask):
    pred_mask = tf.argmax(pred_mask, axis=-1)
    pred_mask = pred_mask[..., tf.newaxis]
    return pred_mask[0]

In [ ]:
def show_predictions(model,dataset=None, num=3):
        for image, mask in dataset.take(1):
            pred_mask = model.predict(image[tf.newaxis, ...])
            for i in range(num):
                plt.figure(figsize=(10, 10))
                plt.subplot(1, 3, 1)
                plt.title('Input Image')
                plt.imshow(image[i])

                plt.subplot(1, 3, 2)
                plt.title('True Mask')
                plt.imshow(mask[1][:, :, 0], cmap='gray')

                plt.subplot(1, 3, 3)
                plt.title('Predicted Mask')
                plt.imshow(create_mask(pred_mask[i,i+1])[:,:,0], cmap='gray')

                plt.show()

In [ ]:
show_predictions(model, train_ds, num=3)

AttributeError: 'Functional' object has no attribute 'take'